In [1]:
print(123)

123


In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
v2 = model.encode(q2)

In [7]:
v2.shape

(384,)

In [8]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [9]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [10]:
v1.dot(dv)

np.float32(0.32332397)

In [11]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [12]:
v2.dot(dv)

np.float32(0.019730574)

In [13]:
from ingest import load_faq_data

documents = load_faq_data()

In [14]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [15]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [16]:
len(texts)

1350

In [17]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [18]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [19]:
vectors[10]

array([-8.04887991e-03, -4.23095562e-02,  6.89805439e-03,  6.72613755e-02,
        1.07818441e-02, -1.00610405e-01,  1.81949940e-02, -4.78682630e-02,
       -1.01894625e-01,  5.02834171e-02, -1.38202179e-02, -3.17115076e-02,
        1.26795014e-02,  9.05294064e-03, -6.50629550e-02,  2.80794520e-02,
        2.61346102e-02, -1.05766185e-01, -2.94326991e-02, -5.09120710e-02,
        4.69526425e-02, -4.12351973e-02,  5.68785286e-03,  1.17212883e-03,
        5.72158284e-02,  6.57509491e-02,  4.00751829e-02,  4.02278192e-02,
        5.93154579e-02, -7.53384782e-03, -7.75877833e-02,  6.71754330e-02,
        6.38393164e-02,  3.66461053e-02,  1.44551853e-02,  2.51878314e-02,
       -1.48494821e-02, -9.69775766e-02, -4.26857881e-02, -1.43325813e-02,
       -3.55788283e-02,  9.69671309e-02,  5.91354556e-02,  3.31453755e-02,
        1.54764121e-02, -6.20953180e-02, -2.13111211e-02, -4.88513708e-02,
        8.62475950e-03,  1.04914397e-01, -3.21247173e-03, -5.28438315e-02,
       -5.83541952e-02, -

In [20]:
vectors[10].shape

(384,)

In [21]:
v1.dot(vectors[10])

np.float32(0.33153272)

In [22]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [23]:
import numpy as np
X = np.array(vectors)

In [24]:
scores = X.dot(v1)

In [25]:
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [26]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [27]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [29]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [31]:
top5

array([  2, 625, 907, 538,   7])

In [32]:
top5 = np.argsort(-scores)[:5]

In [33]:
top5

array([  2, 625, 907, 538,   7])

In [34]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [35]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [36]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [37]:
import os 

GROQ_MODEL = "llama-3.1-8b-instant"
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [38]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [39]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
)

In [40]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still sign up for the course. The key factor is receiving a certificate; to be eligible, you need to submit your project while submissions are still being accepted.'

In [41]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [42]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=groq_client,
)

In [43]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes, you can still join the program. However, if you want to receive a certificate, you need to submit your project while the program is still accepting submissions.'

In [44]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [45]:
vs_index.fit(vectors, documents)

In [46]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [47]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [48]:
vs_index.close()